<a href="https://colab.research.google.com/github/hilaliskandar/censo_senso_rmr/blob/main/RMR_CENSO2022_PIPELINE_COLAB_v0_31.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RMR Censo 2022 — Pipeline Colab v0.31
**Versão operacional obrigatória:** `censo_rmr_colab_v0_27.py` / `0.27.0-colab`.

> Este notebook interrompe a execução se qualquer script antigo for carregado. Execute desde a primeira célula em um runtime reiniciado.


# Censo 2022 — RMR — pipeline operacional no Google Colab

Pipeline incremental e reproduzível para os módulos já validados de equidade/FCU, demografia, composição domiciliar e rendimento, acrescido dos blocos de **condições habitacionais, entorno urbano, densidade demográfica ajustada e integração censitária canônica**.

Princípios: malha canônica como universo territorial; `CD_SETOR` como chave; `X` permanece ausente; módulos em staging; regressão/baseline por bloco; nenhuma promoção automática.


In [1]:
# 1. Montar o Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# 2. Dependências
%pip -q install geopandas pyogrio requests pandas numpy openpyxl scipy


In [3]:
# 3. Localizar automaticamente a versão operacional do script
from pathlib import Path

BASE_DRIVE = Path('/content/drive')
NOME_SCRIPT = 'censo_rmr_colab_v0_27.py'

candidatos = [
    p for p in BASE_DRIVE.rglob(NOME_SCRIPT)
    if p.parent.name == '00_Pipeline'
]

print('Scripts candidatos encontrados:')
for p in candidatos:
    print(' -', p)

if not candidatos:
    raise FileNotFoundError(
        f'{NOME_SCRIPT} não foi encontrado em nenhuma pasta 00_Pipeline do Drive montado.'
    )

if len(candidatos) > 1:
    preferidos = [p for p in candidatos if 'Censo_2022_Setores_RMR' in p.parts]
    if len(preferidos) == 1:
        SCRIPT = preferidos[0]
    else:
        raise RuntimeError('Mais de uma cópia do script foi encontrada: ' + ' | '.join(map(str, candidatos)))
else:
    SCRIPT = candidatos[0]

PIPELINE = SCRIPT.parent
RAIZ = PIPELINE.parent

print()
print('RAIZ encontrada:', RAIZ)
print('PIPELINE:', PIPELINE)
print('SCRIPT:', SCRIPT)
print('Script existe?', SCRIPT.exists())


Scripts candidatos encontrados:
 - /content/drive/MyDrive/protocolo his/Censo_2022_Setores_RMR/00_Pipeline/censo_rmr_colab_v0_27.py

RAIZ encontrada: /content/drive/MyDrive/protocolo his/Censo_2022_Setores_RMR
PIPELINE: /content/drive/MyDrive/protocolo his/Censo_2022_Setores_RMR/00_Pipeline
SCRIPT: /content/drive/MyDrive/protocolo his/Censo_2022_Setores_RMR/00_Pipeline/censo_rmr_colab_v0_27.py
Script existe? True


In [4]:
# 4. Carregar o módulo do Drive com gate de versão
import importlib.util
import sys

EXPECTED_VERSION = '0.27.0-colab'
MODULE_NAME = 'censo_rmr_colab_execucao'

sys.modules.pop(MODULE_NAME, None)

spec = importlib.util.spec_from_file_location(MODULE_NAME, SCRIPT)
if spec is None or spec.loader is None:
    raise RuntimeError(f'Não foi possível carregar o módulo em {SCRIPT}')

censo = importlib.util.module_from_spec(spec)
sys.modules[MODULE_NAME] = censo
spec.loader.exec_module(censo)

versao_carregada = getattr(censo, 'VERSAO_SCRIPT', None)
print('Versão esperada:', EXPECTED_VERSION)
print('Versão carregada:', versao_carregada)
print('Arquivo carregado:', getattr(censo, '__file__', None))

arquivo_carregado = str(getattr(censo, '__file__', ''))
if SCRIPT.name != 'censo_rmr_colab_v0_27.py':
    raise RuntimeError(f'Script incorreto selecionado: {SCRIPT}')
if not arquivo_carregado.endswith('censo_rmr_colab_v0_27.py'):
    raise RuntimeError(f'Módulo incorreto em memória: {arquivo_carregado}')
if versao_carregada != EXPECTED_VERSION:
    raise RuntimeError(
        f'Versão incompatível: esperado {EXPECTED_VERSION}, carregado {versao_carregada}. '
        'Interrompendo para evitar mistura de versões.'
    )

print('Gate de versão: OK')


Versão esperada: 0.27.0-colab
Versão carregada: 0.27.0-colab
Arquivo carregado: /content/drive/MyDrive/protocolo his/Censo_2022_Setores_RMR/00_Pipeline/censo_rmr_colab_v0_27.py
Gate de versão: OK


## Controle metodológico — alfabetização

Os nove indicadores de alfabetização são calculados a partir das **marginais diretas V00900–V00915** do arquivo oficial do IBGE. Cada par representa pessoas alfabetizadas e não alfabetizadas para o universo correspondente. As tabelas cruzadas V008xx não são usadas para estes indicadores, evitando perda de informação por supressões `X`. Valores `X` nas próprias marginais diretas permanecem ausentes e nunca são convertidos em zero.


In [ ]:
# 4A. Pré-voo da versão
assert censo.VERSAO_SCRIPT == '0.27.0-colab'
assert str(censo.__file__).endswith('censo_rmr_colab_v0_27.py'), censo.__file__

ref_equidade = censo.localizar_referencia_equidade(
    RAIZ / '00_Pipeline' / '02_Regressao'
)
ref_densidade = censo.localizar_referencia_densidade(
    RAIZ / '00_Pipeline' / '02_Regressao'
)

print('Pré-voo OK:', censo.VERSAO_SCRIPT)
print('Referência de equidade/FCU:', ref_equidade)
print('Referência de densidade ajustada:', ref_densidade)

if ref_equidade is None:
    raise FileNotFoundError(
        'A referência histórica de equidade/FCU não foi localizada; interrompendo antes do reprocessamento.'
    )
if ref_densidade is None:
    raise FileNotFoundError(
        'A referência histórica de densidade ajustada não foi localizada; interrompendo antes do reprocessamento.'
    )


## Etapa A — auditoria sem reprocessamento

Esta etapa não baixa dados e não recalcula produtos. Ela verifica a estrutura esperada, referências e cache.


In [ ]:
# 5. AUDITAR
import json

auditoria = censo.executar('AUDITAR', RAIZ)
print(json.dumps(auditoria, ensure_ascii=False, indent=2))


## Etapa B — reprocessamento controlado em staging

A célula seguinte pode baixar os arquivos oficiais do IBGE na primeira execução. Nas execuções posteriores, o cache é reutilizado.

O resultado novo ficará somente em:
`00_Pipeline/02_Regressao/v1_colab/equidade_fcu/`


In [ ]:
# 6. REPROCESSAR em staging
resultado = censo.executar(
    'REPROCESSAR_EQUIDADE_FCU_EM_STAGING',
    RAIZ,
    rebaixar_fontes=False,
)
print(json.dumps(resultado, ensure_ascii=False, indent=2))


In [ ]:
# 7. Resumo do teste
reg = resultado.get('regressao_integral', {})
resumo = reg.get('resumo', {})
print('Setores produzidos:', resultado.get('setores_saida'))
print('Âncoras OK:', resultado.get('ancoras', {}).get('ok'))
print('Regressão integral:', reg.get('status'))
print('Regressão OK:', resumo.get('ok'))
print('Chaves apenas no novo:', resumo.get('chaves_apenas_novo'))
print('Chaves apenas na referência:', resumo.get('chaves_apenas_referencia'))
print('Divergências numéricas:', resumo.get('divergencias_numericas'))
print('Divergências textuais:', resumo.get('divergencias_textuais'))
print('Promoção permitida:', resultado.get('promocao_permitida'))


In [ ]:
# 8. Onde os produtos foram gravados
from pathlib import Path
staging = RAIZ / '00_Pipeline/02_Regressao/v1_colab'
for p in sorted(staging.rglob('*')):
    if p.is_file():
        print(p.relative_to(RAIZ))


## Critério para considerar o primeiro ensaio bem-sucedido

O teste não exige que a regressão histórica seja idêntica de saída. O que precisa acontecer é:

1. a execução terminar sem erro de contrato ou de fonte;
2. o produto conter o universo RMR esperado ou uma diferença explicitamente diagnosticada;
3. as cinco âncoras serem avaliadas;
4. a comparação integral informar chaves e divergências por coluna;
5. `promocao_permitida` permanecer `False`.

Se houver divergências, preserve o `RELATORIO_REGRESSAO_EQUIDADE_FCU.json`; ele é a evidência para a próxima rodada de diagnóstico.


## Etapa C — demografia e estrutura etária

Este bloco é independente do módulo de equidade já validado. A malha de 7.208 setores permanece como universo territorial. O arquivo temático de demografia entra por `left join`; setores ausentes do tema e valores `X` permanecem ausentes.

Controles: identidades entre total, sexo e 11 grupos etários; cinco âncoras setoriais; seis âncoras municipais do Bloco 1 histórico; nenhuma promoção automática.


In [ ]:
# 9. REPROCESSAR DEMOGRAFIA em staging
demografia = censo.executar(
    'REPROCESSAR_DEMOGRAFIA_EM_STAGING',
    RAIZ,
    rebaixar_fontes=False,
)
print(json.dumps(demografia, ensure_ascii=False, indent=2))


In [ ]:
# 10. Resumo do teste demográfico
u = demografia.get('universo', {})
inv = demografia.get('invariantes_contabeis', {})
anc = demografia.get('ancoras_setoriais', {})
anc_m = demografia.get('ancoras_municipais_historicas', {})
print('Setores produzidos:', demografia.get('setores_saida'))
print('Presentes no tema demografia:', u.get('setores_presentes_no_tema_demografia'))
print('Ausentes do tema demografia:', u.get('setores_ausentes_do_tema_demografia'))
print('POP_TOTAL suprimido em linhas presentes:', u.get('setores_presentes_com_POP_TOTAL_suprimido'))
print('Setores sem POP_TOTAL no produto canônico:', u.get('setores_sem_POP_TOTAL_no_produto_canonico'))
print('Invariantes contábeis OK:', inv.get('ok'))
print('Âncoras setoriais OK:', anc.get('ok'))
print('Âncoras municipais históricas OK:', anc_m.get('ok'))
print('Baseline integral materializada:', demografia.get('baseline_integral_materializada'))
print('Promoção permitida:', demografia.get('promocao_permitida'))


In [ ]:
# 11. Produtos demográficos em staging
for nome, caminho in demografia.get('produtos', {}).items():
    print(nome + ':', Path(caminho).relative_to(RAIZ))
print('relatório:', Path(RAIZ / '00_Pipeline/02_Regressao/v1_colab/RELATORIO_VALIDACAO_DEMOGRAFIA.json').relative_to(RAIZ))


### Critério para fechar o teste demográfico

Esperado nesta versão: 7.208 setores no produto canônico, 7.150 linhas temáticas, 58 setores sem linha no arquivo de demografia, 54 linhas temáticas com `POP_TOTAL` suprimido, zero violações nas identidades contábeis e aprovação das âncoras setoriais e municipais. A baseline integral ainda não é promovida automaticamente.


## Etapa D — composição das unidades domésticas

Este bloco usa o arquivo oficial de parentesco e composição doméstica. Domicílios coletivos são excluídos dos denominadores de situação conjugal, composição com filhos e espécie da unidade doméstica. Valores `X` permanecem ausentes. O resultado é gravado somente em staging.


In [ ]:
# 8. REPROCESSAR composição domiciliar em staging
resultado_comp = censo.executar(
    'REPROCESSAR_COMPOSICAO_DOMICILIAR_EM_STAGING',
    RAIZ,
    rebaixar_fontes=False,
)
print(json.dumps(resultado_comp, ensure_ascii=False, indent=2))


In [ ]:
# 9. Resumo do teste de composição domiciliar
u = resultado_comp.get('universo', {})
print('Setores produzidos:', resultado_comp.get('setores_saida'))
print('Presentes no tema parentesco:', u.get('setores_presentes_no_tema_parentesco'))
print('Ausentes do tema parentesco:', u.get('setores_ausentes_do_tema_parentesco'))
print('Setores válidos para heterogeneidade:', u.get('setores_heterogeneidade_validos'))
print('Invariantes contábeis OK:', resultado_comp.get('invariantes_contabeis', {}).get('ok'))
print('Âncoras municipais históricas OK:', resultado_comp.get('ancoras_municipais_historicas', {}).get('ok'))
print('Baseline integral materializada:', resultado_comp.get('baseline_integral_materializada'))
print('Promoção permitida:', resultado_comp.get('promocao_permitida'))
print('00_Pipeline/02_Regressao/v1_colab/RELATORIO_VALIDACAO_COMPOSICAO_DOMICILIAR.json')
print('00_Pipeline/02_Regressao/v1_colab/composicao_domiciliar/RMR_CENSO2022_COMPOSICAO_DOMICILIAR_SETOR.csv')


## 8. Rendimento da pessoa responsável

Usa a atualização oficial IBGE de 08/05/2026. `V06004` é a média, `V06005` a variância e `V06006` a mediana do rendimento nominal mensal das pessoas responsáveis com rendimento. O módulo não interpreta esses campos como renda domiciliar, renda per capita ou pobreza. `X` permanece ausente.


In [ ]:
# 8A. Reprocessar rendimento em staging
rel_renda = censo.executar(
    "REPROCESSAR_RENDIMENTO_EM_STAGING",
    RAIZ,
    rebaixar_fontes=False,
)

print("Setores produzidos:", rel_renda["setores_saida"])
print("Presentes no tema rendimento:", rel_renda["universo"]["setores_presentes_no_tema_rendimento"])
print("Ausentes do tema rendimento:", rel_renda["universo"]["setores_ausentes_do_tema_rendimento"])
print("Setores estáveis (>=20 responsáveis):", rel_renda["universo"]["setores_renda_estavel_resp_ge20"])
print("Invariantes contábeis OK:", rel_renda["invariantes_contabeis"]["ok"])
print("Âncoras setoriais históricas OK:", rel_renda["ancoras_setoriais_historicas"]["ok"])
print("Baseline integral materializada:", rel_renda["baseline_integral_materializada"])
print("Promoção permitida:", rel_renda["promocao_permitida"])
print(rel_renda["produto"]["setorial"])


## Etapa E — condições habitacionais e saneamento domiciliar

O núcleo histórico é reconstruído diretamente dos agregados oficiais de **Características do Domicílio — Partes 1 e 2**, usando a Parte 2 corrigida em 17/04/2025. A Parte 3 corrigida em 17/04/2025 permanece registrada como fonte auditada, mas não é necessária para reproduzir as quatro dimensões históricas nesta etapa.

Dimensões reproduzidas:
- física: domicílios particulares improvisados ocupados / total de domicílios particulares ocupados;
- água: DPPO sem água encanada / DPPO;
- sanitária: destinos precários de esgoto `V00312–V00316` / DPPO;
- resíduos: destinos inadequados `V00399–V00402` / DPPO;
- pressão de ocupação: DPPO com seis ou mais moradores / DPPO.

Indicadores adicionais como cortiço, estrutura degradada, banheiro comum e ausência de ligação à rede de água permanecem **separados**, para evitar dupla contagem. Os resultados são proxies de precariedade/inadequação e não equivalem ao déficit habitacional oficial.


**Revisão operacional v0.13 — dimensões históricas de moradia.** Para reproduzir a matriz histórica, as dimensões são compostas pela pior condição observável entre indicadores atômicos do mesmo domínio: física = máximo entre improvisado, cortiço e estrutura degradada/inacabada; água = máximo entre ausência de canalização intradomiciliar, ausência de ligação à rede geral e fonte precária (carro-pipa ou rios/açudes/córregos/lagos); sanitária = máximo entre instalação sanitária precária e esgotamento precário. Resíduos e 6+ moradores usam a soma das parcelas publicadas. Valores `X` permanecem ausentes nas variáveis originais. Quando uma soma composta contém parcelas suprimidas, o resultado é documentado como **limite inferior observável**, e não como imputação de zero.


In [ ]:
# 12. REPROCESSAR MORADIA em staging
rel_moradia = censo.executar(
    "REPROCESSAR_MORADIA_EM_STAGING",
    RAIZ,
    rebaixar_fontes=False,
)

u = rel_moradia.get("universo", {})
print("Setores produzidos:", rel_moradia.get("setores_saida"))
print("Presentes em Domicílio 1:", u.get("presentes_domicilio1"))
print("Ausentes em Domicílio 1:", u.get("ausentes_domicilio1"))
print("Presentes em Domicílio 2:", u.get("presentes_domicilio2"))
print("Ausentes em Domicílio 2:", u.get("ausentes_domicilio2"))
print("Setores estáveis (>=20 DPPO):", u.get("setores_estaveis_dppo_ge20"))
print("Invariantes contábeis OK:", rel_moradia.get("invariantes_contabeis", {}).get("ok"))
print("Âncoras setoriais históricas OK:", rel_moradia.get("ancoras_setoriais_historicas", {}).get("ok"))
print("Baseline integral materializada:", rel_moradia.get("baseline_integral_materializada"))
print("Promoção permitida:", rel_moradia.get("promocao_permitida"))
for nome, caminho in rel_moradia.get("produtos", {}).items():
    print(nome + ':', Path(caminho).relative_to(RAIZ))
print('relatório:', Path(RAIZ / '00_Pipeline/02_Regressao/v1_colab/RELATORIO_VALIDACAO_MORADIA.json').relative_to(RAIZ))


## Etapa F — entorno urbano

Este bloco usa como camada canônica os **agregados absolutos oficiais de moradores do entorno, divulgados em abril de 2025**, preservando os cálculos contínuos que reproduzem a série histórica. A publicação percentual oficial posterior, de dezembro de 2025, entra como camada de QA e versionamento.

O universo canônico continua sendo a malha de 7.208 setores. O universo temático do entorno é menor. `V05200` representa moradores no universo do entorno. O P80 histórico é calculado somente entre setores com `V05200 >= 20`. A classificação permanece exploratória, relativa à RMR e **não constitui padrão normativo de adequação urbana**.

A auditoria registra separadamente revisões oficiais posteriores em arborização, obstáculo na calçada e rampa; essas revisões não são forçadas a coincidir com a série histórica.


In [ ]:
# 13. REPROCESSAR ENTORNO em staging
rel_entorno = censo.executar(
    "REPROCESSAR_ENTORNO_EM_STAGING",
    RAIZ,
    rebaixar_fontes=False,
)

u = rel_entorno.get("universo", {})
print("Setores produzidos:", rel_entorno.get("setores_saida"))
print("Setores cobertos pelo entorno:", u.get("setores_cobertos_entorno"))
print("Setores fora/sem registro no entorno:", u.get("setores_fora_ou_sem_registro_entorno"))
print("Setores estáveis (V05200 >= 20):", u.get("setores_estaveis_entorno_v05200_ge20"))
print("Válidos por indicador:", u.get("validos_por_indicador"))
print("Âncoras históricas de lógica OK:", rel_entorno.get("ancoras_setoriais_historicas", {}).get("ok"))
print("Consistência estrita série absoluta × publicação percentual posterior OK:", rel_entorno.get("consistencia_publicacao_oficial_atual_ok"))
print("Consistência dos indicadores não revisados OK:", rel_entorno.get("consistencia_indicadores_nao_revisados_ok"))
print("Revisões oficiais percentuais detectadas:", rel_entorno.get("revisoes_oficiais_percentuais_detectadas"))
reg = rel_entorno.get("regressao_integral_historica", {}).get("resumo", {})
print("Regressão histórica exata OK:", reg.get("ok"))
print("Revisão oficial detectada:", rel_entorno.get("revisao_oficial_detectada"))
print("Validação do módulo de entorno OK:", rel_entorno.get("validacao_modulo_entorno_ok"))
print("Chaves apenas no novo:", reg.get("chaves_apenas_novo"))
print("Chaves apenas na referência:", reg.get("chaves_apenas_referencia"))
print("Divergências numéricas históricas:", reg.get("divergencias_numericas"))
print("Divergências textuais históricas:", reg.get("divergencias_textuais"))
print("Baseline integral materializada:", rel_entorno.get("baseline_integral_materializada"))
print("Promoção permitida:", rel_entorno.get("promocao_permitida"))
for nome, caminho in rel_entorno.get("produtos", {}).items():
    print(nome + ':', Path(caminho).relative_to(RAIZ))
print('relatório:', Path(RAIZ / '00_Pipeline/02_Regressao/v1_colab/RELATORIO_VALIDACAO_ENTORNO.json').relative_to(RAIZ))


## Etapa G — área efetivamente domiciliada e densidade demográfica ajustada

O módulo utiliza diretamente o produto oficial do IBGE **Área territorial efetivamente domiciliada e densidade demográfica ajustada dos Setores Censitários**. `AREA_DOM` não é reconstruída a partir da geometria da malha. O IBGE identifica a área efetivamente domiciliada a partir da concentração dos endereços recenseados em grade regular de 250 m e divulga esse atributo apenas para situações territoriais urbanas (`CD_SIT` 1, 2 e 3).

Campos canônicos: `AREA_DOMICILIADA_KM2`, `AREA_KM2`, `DENSIDADE_DEMOGRAFICA_DOMICILIADA_HAB_KM2` e `DENSIDADE_DEMOGRAFICA_SETOR_HAB_KM2`. A densidade é tratada como **contexto territorial**, não como sinal automático de vulnerabilidade. A agregação municipal é calculada por soma do numerador e das áreas, nunca por média simples das densidades setoriais.


In [ ]:
# 14. REPROCESSAR DENSIDADE AJUSTADA em staging
rel_dens = censo.executar(
    "REPROCESSAR_DENSIDADE_AJUSTADA_EM_STAGING",
    RAIZ,
    rebaixar_fontes=False,
)

u = rel_dens.get("universo", {})
qa = rel_dens.get("qa_fonte_oficial", {})
reg = rel_dens.get("regressao_integral_historica", {}).get("resumo", {})
reg_fcu_hist = rel_dens.get("comparacao_contextual_fcu_referencia_densidade", {}).get("resumo", {})
reg_fcu_eq = rel_dens.get("consistencia_fcu_referencia_equidade", {}).get("resumo", {})
print("Setores produzidos:", rel_dens.get("setores_saida"))
print("Setores com AREA_DOM oficial:", u.get("setores_com_area_domiciliada"))
print("Setores sem AREA_DOM oficial:", u.get("setores_sem_area_domiciliada"))
print("QA da fonte oficial OK:", qa.get("ok"))
print("Âncoras setoriais históricas OK:", rel_dens.get("ancoras_setoriais_historicas", {}).get("ok"))
print("Regressão histórica do núcleo de densidade OK:", reg.get("ok"))
print("Chaves apenas no novo:", reg.get("chaves_apenas_novo"))
print("Chaves apenas na referência:", reg.get("chaves_apenas_referencia"))
print("Divergências numéricas no núcleo:", reg.get("divergencias_numericas"))
print("Divergências textuais no núcleo:", reg.get("divergencias_textuais"))
print("Comparação contextual FCU da referência de densidade OK:", reg_fcu_hist.get("ok"))
print("Divergências contextuais FCU na referência de densidade:", reg_fcu_hist.get("divergencias_numericas"))
print("Consistência FCU com a referência de equidade OK:", reg_fcu_eq.get("ok"))
print("Validação do módulo de densidade OK:", rel_dens.get("validacao_modulo_densidade_ok"))
print("Baseline integral materializada:", rel_dens.get("baseline_integral_materializada"))
print("Promoção permitida:", rel_dens.get("promocao_permitida"))
for nome, caminho in rel_dens.get("produtos", {}).items():
    print(nome + ':', Path(caminho).relative_to(RAIZ))
print('relatório:', Path(RAIZ / '00_Pipeline/02_Regressao/v1_colab/RELATORIO_VALIDACAO_DENSIDADE_AJUSTADA.json').relative_to(RAIZ))


## Etapa H — integração da base censitária canônica

Esta etapa consolida os sete produtos setoriais validados em uma única base de 7.208 setores. Os campos compartilhados de identificação, população, FCU e área são comparados setor a setor antes da junção; qualquer divergência bloqueia o processo. Supressões e ausências temáticas permanecem `NA`, sem conversão para zero.

O produto integrado é gravado em staging juntamente com um inventário técnico das colunas e um relatório de QA. A etapa não promove automaticamente a base para produtos históricos e não inicia interpretações substantivas.

In [ ]:
# 15. CONSOLIDAR BASE CENSITARIA em staging
rel_integracao = censo.executar(
    "CONSOLIDAR_BASE_CENSITARIA_EM_STAGING",
    RAIZ,
)

estrutura = rel_integracao.get("checks_estrutura_canonica", {})
percentuais = rel_integracao.get("checks_percentuais", {})
nao_neg = rel_integracao.get("checks_nao_negatividade", {})
pop = rel_integracao.get("coerencia_populacao_demografia_densidade", {})
dens = rel_integracao.get("checks_formulas_densidade", {})

print("Setores integrados:", rel_integracao.get("setores_saida"))
print("Colunas integradas:", rel_integracao.get("colunas_saida"))
print("Universo de 7.208 setores OK:", estrutura.get("universo_7208", {}).get("ok"))
print("Chave setorial única OK:", estrutura.get("chave_unica", {}).get("ok"))
print("Quatorze municípios OK:", estrutura.get("quatorze_municipios", {}).get("ok"))
print("Percentuais no intervalo 0-100 OK:", percentuais.get("ok"))
print("Campos não negativos OK:", nao_neg.get("ok"))
print("População demografia x densidade OK:", pop.get("ok"))
print("Casos populacionais comparados:", pop.get("ambos_preenchidos"))
print("Fórmulas de densidade OK:", dens.get("ok"))
print("Validação da base integrada OK:", rel_integracao.get("validacao_base_censitaria_integrada_ok"))
print("Baseline integral materializada:", rel_integracao.get("baseline_integral_materializada"))
print("Promoção permitida:", rel_integracao.get("promocao_permitida"))
for nome, caminho in rel_integracao.get("produtos", {}).items():
    print(nome + ":", Path(caminho).relative_to(RAIZ))
print("relatório:", Path(RAIZ / "00_Pipeline/02_Regressao/v1_colab/RELATORIO_VALIDACAO_BASE_CENSITARIA_INTEGRADA.json").relative_to(RAIZ))

if not rel_integracao.get("validacao_base_censitaria_integrada_ok"):
    raise RuntimeError("O gate da base censitária integrada não foi aprovado.")


## Etapa I — materialização controlada da baseline integral

Esta etapa só é executada após o gate integral da Etapa H. Ela copia a base integrada, o inventário técnico e o relatório de validação para `00_Pipeline/02_Regressao/baseline_integral/v1`, registra hashes SHA-256 e cria um manifesto de proveniência. A operação é idempotente e bloqueia qualquer sobrescrita divergente.

A materialização da baseline **não equivale à promoção para produtos finais** e não inicia análises espaciais derivadas.


In [ ]:
# 16. MATERIALIZAR BASELINE INTEGRAL validada
if not rel_integracao.get("validacao_base_censitaria_integrada_ok"):
    raise RuntimeError("Materialização bloqueada: o gate integral da Etapa H não foi aprovado.")

rel_baseline = censo.executar(
    "MATERIALIZAR_BASELINE_INTEGRAL",
    RAIZ,
)

print("Baseline ID:", rel_baseline.get("baseline_id"))
print("Baseline integral materializada:", rel_baseline.get("baseline_integral_materializada"))
print("Baseline reutilizada:", rel_baseline.get("baseline_reutilizada"))
print("Setores:", rel_baseline.get("setores"))
print("Colunas:", rel_baseline.get("colunas"))
print("Promoção para produtos finais:", rel_baseline.get("promocao_para_produtos_finais"))
for nome, caminho in rel_baseline.get("produtos_baseline", {}).items():
    print(nome + ":", Path(caminho).relative_to(RAIZ))
print("manifesto:", Path(rel_baseline.get("manifesto")).relative_to(RAIZ))
print("Hashes SHA-256:")
for nome, valor in rel_baseline.get("hashes_baseline", {}).items():
    print(" -", nome, valor)

if rel_baseline.get("baseline_integral_materializada") is not True:
    raise RuntimeError("A baseline integral não foi materializada.")
if rel_baseline.get("setores") != 7208 or rel_baseline.get("colunas") != 166:
    raise RuntimeError("A baseline materializada não corresponde ao gate integral esperado.")


## Etapa J — matriz analitica candidata e auditoria de redundancia

Esta etapa consome exclusivamente a baseline integral `RMR_CENSO2022_BASELINE_INTEGRAL_v1`. Ela nao altera os artefatos congelados e nao executa PCA, clustering, imputacao, ranking ou indice global.

A matriz separa atributos territoriais, demografia, composicao domiciliar, equidade, condicao socioeconomica, moradia e entorno. Densidade permanece como atributo territorial; composicao racial permanece fora de escores de deficiencia; indicadores sinteticos preexistentes ficam fora da PCA atomica para evitar dupla contagem.

O limiar `|r| >= 0,85` serve apenas para sinalizar possivel redundancia. Nenhuma variavel e eliminada automaticamente. Com a baseline ja materializada, esta etapa pode ser executada diretamente apos as celulas de montagem, dependencias e carregamento/gate da versao.


In [ ]:
# 17. CONSTRUIR MATRIZ ANALITICA candidata
rel_matriz = censo.executar(
    "CONSTRUIR_MATRIZ_ANALITICA_EM_STAGING",
    RAIZ,
)

print("Baseline:", rel_matriz.get("baseline_id"))
print("Setores:", rel_matriz.get("setores"))
print("Municipios:", rel_matriz.get("municipios"))
print("Colunas da matriz:", rel_matriz.get("colunas_matriz"))
print("Variaveis especificadas:", rel_matriz.get("variaveis_especificadas"))
print("Candidatas a PCA:", rel_matriz.get("variaveis_pca_candidatas"))
print("Pares sinalizados por redundancia:", rel_matriz.get("pares_sinalizados_redundancia"))
for nome, ok in rel_matriz.get("gates", {}).items():
    print(nome + ":", ok)
print("Promocao para produtos finais:", rel_matriz.get("promocao_para_produtos_finais"))
print("Produtos:")
for nome, caminho in rel_matriz.get("produtos", {}).items():
    print(" -", nome, Path(caminho).relative_to(RAIZ))


## Etapa K — auditoria pre-PCA

Esta etapa consome os produtos validados da Etapa J e ainda **nao executa PCA nem clustering**. Ela mede cobertura, missingness, estatisticas descritivas, correlacoes pareadas, impacto de cada variavel sobre a amostra completa e adequacao preliminar pela estatistica KMO e pelo teste de esfericidade de Bartlett.

As ausencias permanecem `NA`; nao ha imputacao. A unica exclusao substantiva proposta automaticamente e condicionada ao par sanitário conceitualmente aninhado já identificado na Etapa J: caso `PCT_APENAS_SANITARIO_BURACO` e `PCT_SEM_BANHEIRO_EXCLUSIVO_COMPLETO` mantenham `|r| >= 0,85`, o primeiro permanece na base descritiva, mas e retirado da matriz PCA recomendada para evitar dupla ponderacao da mesma dimensao.

KMO/MSA e Bartlett sao tratados como diagnosticos, nao como criterios isolados de decisao.


In [ ]:
# 18. AUDITAR PRE-PCA em staging
rel_pre_pca = censo.executar(
    "AUDITAR_PRE_PCA_EM_STAGING",
    RAIZ,
)

print("Baseline:", rel_pre_pca.get("baseline_id"))
print("Setores:", rel_pre_pca.get("setores"))
print("Candidatas iniciais:", rel_pre_pca.get("candidatas_iniciais"))
print("Candidatas recomendadas:", rel_pre_pca.get("candidatas_recomendadas"))
print("Casos completos - 20 variaveis:", rel_pre_pca.get("casos_completos_20"))
print("Casos completos - matriz recomendada:", rel_pre_pca.get("casos_completos_recomendados"))
print("Percentual de casos completos recomendado:", rel_pre_pca.get("pct_casos_completos_recomendados"))
print("Exclusao por aninhamento sanitario aplicada:", rel_pre_pca.get("exclusao_aninhamento_sanitario_aplicada"))
print("Pares redundantes:", len(rel_pre_pca.get("pares_redundantes", [])))

diag = rel_pre_pca.get("kmo_bartlett_recomendado", {})
print("KMO/Bartlett disponivel:", diag.get("disponivel"))
print("KMO global recomendado:", diag.get("kmo_global"))
print("Bartlett:", diag.get("bartlett"))

for nome, ok in rel_pre_pca.get("gates", {}).items():
    print(f"{nome}: {ok}")

print("Promocao para produtos finais:", rel_pre_pca.get("promocao_para_produtos_finais"))
print("Produtos:")
for nome, caminho in rel_pre_pca.get("produtos", {}).items():
    print(" -", nome, caminho)


## Etapa L — PCA exploratória e diagnóstico da amostra
Executa PCA somente nos 2.586 casos completos esperados da matriz recomendada de 19 variáveis. Não imputa, não faz clustering e não promove produtos finais.


In [ ]:
# 19. EXECUTAR PCA exploratória em staging
rel_pca = censo.executar(
    'EXECUTAR_PCA_EXPLORATORIA_EM_STAGING',
    RAIZ,
)
print('Baseline:', rel_pca.get('baseline_id'))
print('Setores universo:', rel_pca.get('setores_universo'))
print('Variáveis PCA:', rel_pca.get('variaveis_pca'))
print('Casos completos:', rel_pca.get('casos_completos'))
print('Percentual casos completos:', rel_pca.get('pct_casos_completos'))
print('Componentes Kaiser:', rel_pca.get('componentes_kaiser'))
print('Componentes análise paralela p95:', rel_pca.get('componentes_analise_paralela_p95'))
print('Componentes retidos:', rel_pca.get('componentes_retidos_para_interpretacao'))
print('Variância acumulada retida (%):', rel_pca.get('variancia_acumulada_retida_pct'))
print('PCA exploratória concluída:', (rel_pca.get('gates') or {}).get('pca_exploratoria_concluida'))
print('Promoção para produtos finais:', rel_pca.get('promocao_para_produtos_finais'))
print('Produtos:')
for k,v in (rel_pca.get('produtos') or {}).items(): print(' -', k, v)


In [ ]:
# 20. Visualizar scree + análise paralela, se a figura tiver sido gerada
from IPython.display import display, Image
png = (rel_pca.get('produtos') or {}).get('scree_png')
if png:
    display(Image(filename=png))


## Etapa M — Auditoria pós-PCA

Audita a solução de quatro componentes antes de qualquer clustering. Examina comunalidades, cargas cruzadas, interpretabilidade, cobertura municipal e diferenças padronizadas entre setores incluídos e excluídos. Não recalcula PCA, não imputa dados e não executa clustering.


In [5]:
# 20. AUDITAR a solução PCA antes das tipologias
rel_pos_pca = censo.executar(
    "AUDITAR_POS_PCA_EM_STAGING",
    RAIZ,
)

print("Baseline:", rel_pos_pca.get("baseline_id"))
print("Componentes auditados:", rel_pos_pca.get("componentes_auditados"))
print("Casos nos scores:", rel_pos_pca.get("casos_scores"))
print("Variáveis auditadas:", rel_pos_pca.get("variaveis_auditadas"))
print("Baixa comunalidade (<0,30):", rel_pos_pca.get("baixa_comunalidade_lt_030"))
print("Cargas cruzadas (|carga|>=0,40 em >=2 PCs):", rel_pos_pca.get("cargas_cruzadas_abs_ge_040"))
print("Representação fraca (máx |carga|<0,40):", rel_pos_pca.get("representacao_fraca_max_abs_lt_040"))
print("Cobertura municipal mínima (%):", rel_pos_pca.get("cobertura_municipal_min_pct"))
print("Cobertura municipal máxima (%):", rel_pos_pca.get("cobertura_municipal_max_pct"))
print("CV da cobertura municipal:", rel_pos_pca.get("cobertura_municipal_cv"))
print("Variáveis com |SMD|>=0,20:", rel_pos_pca.get("variaveis_com_abs_smd_ge_020"))
print("Variáveis com |SMD|>=0,50:", rel_pos_pca.get("variaveis_com_abs_smd_ge_050"))
print("Decisão:", rel_pos_pca.get("decisao"))
print("Auditoria pós-PCA concluída:", rel_pos_pca.get("gates", {}).get("auditoria_pos_pca_concluida"))
print("Promoção para produtos finais:", rel_pos_pca.get("promocao_para_produtos_finais"))
print("Produtos:")
for k, v in rel_pos_pca.get("produtos", {}).items():
    print(" -", k, v)


Baseline: RMR_CENSO2022_BASELINE_INTEGRAL_v1
Componentes auditados: 4
Casos nos scores: 2586
Variáveis auditadas: 19
Baixa comunalidade (<0,30): 5
Cargas cruzadas (|carga|>=0,40 em >=2 PCs): 2
Representação fraca (máx |carga|<0,40): 3
Cobertura municipal mínima (%): 26.47058823529412
Cobertura municipal máxima (%): 41.9773095623987
CV da cobertura municipal: 0.14413822202775373
Variáveis com |SMD|>=0,20: 4
Variáveis com |SMD|>=0,50: 0
Decisão: {'quatro_componentes_interpretaveis': True, 'scores_aptos_para_tipologias_dos_casos_completos': True, 'generalizacao_direta_ao_universo_7208': False, 'exige_analise_sensibilidade_missingness_antes_de_generalizar': True, 'clustering_executado': False}
Auditoria pós-PCA concluída: True
Promoção para produtos finais: False
Produtos:
 - diagnostico_variaveis /content/drive/MyDrive/protocolo his/Censo_2022_Setores_RMR/00_Pipeline/04_Analise_Espacial/04_Auditoria_Pos_PCA/v1/RMR_CENSO2022_POS_PCA_DIAGNOSTICO_VARIAVEIS.csv
 - diagnostico_componentes /con